# Section VI-B: Error Mitigation for Five-Qubit Circuits
**Fig. 20** — QFT, GraphState, QuantumVolume on up to two sets of 5 physical qubits with/without single-shot OTEM.

> Qubit layouts are selected automatically via OTEM preprocessing (BFS over the device coupling map).
> The paper used `ibm_torino` with physical qubits `[86,87,88,89,90]` and `[117,116,115,114,129]`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # qatg package
sys.path.insert(0, os.path.abspath('.'))    # otem.py, result_analyze.py

In [ ]:
from collections import deque
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile, ClassicalRegister
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.circuit.library import QFT, GraphState, QuantumVolume
from otem import OTEM
from result_analyze import srate   # srate(counts, nqb) = P(all-zeros)

## 1. Connect to backend

In [ ]:
service = QiskitRuntimeService(
    channel='ibm_cloud',
    instance='YOUR_IBMQ_INSTANCE_HERE',
    token='YOUR_IBMQ_TOKEN_HERE'
)
backend_name = 'YOUR_BACKEND_NAME'   # e.g. 'ibm_torino', 'ibm_boston'
backend  = service.backend(backend_name)
nqb      = backend.num_qubits
sampler  = Sampler(backend)
shots    = 10000
print(f"Backend: {backend.name}  ({nqb} qubits)")

## 2. Load compressed test circuits & preprocessing

In [ ]:
ltid = [0, 1, 2, 3, 4, 5, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
test_datas = [QuantumCircuit.from_qasm_file(f'test_circuits/test_circuit_{i}.qasm')
              for i in ltid]
print(f"Loaded {len(test_datas)} test circuits")

my_em  = OTEM(backend, test_datas)
preprocess_qcs = my_em.build_preprocess_test()

job_pre    = sampler.run(preprocess_qcs, shots=shots)
print('Preprocess job:', job_pre.job_id())
pre_result = job_pre.result()
max_ac_qb, _, qb_f_ac, test_id = my_em.qubit_and_online_test_selection_from_result(pre_result)
print(f"Most faulty qubit: {max_ac_qb}  |  test_id assigned to {sum(t != -1 for t in test_id)} qubits")

## 3. Find qubit layouts automatically (BFS)
Starting from the highest-scoring faulty qubit, BFS expands to neighboring physical qubits
until a connected layout of `N_QUBITS` qubits is found. Up to `MAX_LAYOUTS` non-overlapping
layouts are collected.

In [ ]:
N_QUBITS    = 5
MAX_LAYOUTS = 2


def find_connected_layouts(backend, test_id, qb_f_ac, n_qubits=N_QUBITS, max_layouts=MAX_LAYOUTS):
    """BFS from highest-scoring faulty qubits; returns up to max_layouts
    non-overlapping connected physical qubit lists of length n_qubits."""
    adj = {q: set() for q in range(backend.num_qubits)}
    for a, b in backend.coupling_map.get_edges():
        adj[a].add(b); adj[b].add(a)

    faulty = sorted(
        [(qb, qb_f_ac[qb]) for qb in range(len(test_id)) if test_id[qb] != -1],
        key=lambda x: -x[1],
    )
    print(f"Faulty qubits (sorted by score): {[qb for qb, _ in faulty]}")

    used, layouts = set(), []
    for center, score in faulty:
        if center in used:
            continue
        visited = {center}
        queue   = deque([center])
        layout  = [center]
        while queue and len(layout) < n_qubits:
            node = queue.popleft()
            for nb in sorted(adj[node]):
                if nb not in visited and nb not in used:
                    visited.add(nb); queue.append(nb); layout.append(nb)
                    if len(layout) >= n_qubits:
                        break
        if len(layout) < n_qubits:
            print(f"  Skipping qubit {center}: only {len(layout)} free connected qubits")
            continue
        layout = sorted(layout)
        used.update(layout)
        layouts.append(layout)
        print(f"  Layout {len(layouts)}: physical qubits {layout}  (center={center}, score={score:.4f})")
        if len(layouts) >= max_layouts:
            break

    if not layouts:
        raise RuntimeError("No valid layout found — no faulty qubits detected during preprocessing.")
    return layouts


def build_ot_for_layout(test_datas, test_id, layout):
    """Build an N_QUBITS-wide OT circuit for the given physical layout.
    Local index i ↔ layout[i]; identity on qubits with test_id == -1."""
    qc = QuantumCircuit(len(layout))
    for local_idx, qb_phys in enumerate(layout):
        tid = test_id[qb_phys] if qb_phys < len(test_id) else -1
        if tid != -1:
            q = test_datas[tid].copy()
            q.remove_final_measurements()
            for inst in q.data:
                qc.append(inst.operation, [local_idx])
    return qc


ALL_LAYOUTS = find_connected_layouts(backend, test_id, qb_f_ac)
ot_circuits = [build_ot_for_layout(test_datas, test_id, layout) for layout in ALL_LAYOUTS]
N_LAY = len(ALL_LAYOUTS)
print(f"\nFound {N_LAY} layout(s): {ALL_LAYOUTS}")

## 4. Build benchmark circuits
Each circuit is `forward ∘ inverse` so the ideal output is |0…0⟩ (P = 1).
No coupling-map constraint here; hardware routing is handled at transpile time.

In [ ]:
ct_list = ['QFT', 'GraphState', 'QuantumVolume']


def build_5qb_circuits():
    c = {}

    qft = QFT(N_QUBITS)
    qc = QuantumCircuit(N_QUBITS)
    qc.append(qft, range(N_QUBITS)); qc.barrier()
    qc.append(qft.inverse(), range(N_QUBITS))
    c['QFT'] = qc

    gs = GraphState([[1] * N_QUBITS] * N_QUBITS)
    qc2 = QuantumCircuit(N_QUBITS)
    qc2.compose(gs, inplace=True); qc2.barrier()
    qc2.compose(gs.inverse(), inplace=True)
    c['GraphState'] = qc2

    qv = QuantumVolume(N_QUBITS)
    qc3 = QuantumCircuit(N_QUBITS)
    qc3.append(qv, range(N_QUBITS)); qc3.barrier()
    qc3.append(qv.inverse(), range(N_QUBITS))
    c['QuantumVolume'] = qc3

    return c


bench = build_5qb_circuits()
print(f"Benchmark circuits: {list(bench.keys())}")

## 5. Run experiments per layout
For each layout:
- **Without OTEM**: algorithm + measure (`meas` register)
- **With OTEM**: OT + measure(`testresult`) + algorithm + measure(`measureresult`)

Both use `optimization_level=1` so the transpiler can route within the (non-necessarily-linear) BFS layout.

In [ ]:
def build_no_otem_circuit(qc_alg, layout, backend):
    n = len(layout)
    qc = QuantumCircuit(n)
    cr = ClassicalRegister(n, name='meas')
    qc.add_register(cr)
    qc.compose(qc_alg, inplace=True)
    qc.measure(range(n), cr)
    return transpile(qc, backend, initial_layout=layout, optimization_level=1)


def build_with_otem_circuit(qc_alg, qc_ot, layout, backend):
    n = len(layout)
    qc = QuantumCircuit(n)
    qc.compose(qc_ot, inplace=True)
    cr_t = ClassicalRegister(n, name='testresult')
    qc.add_register(cr_t)
    qc.measure(range(n), cr_t)
    qc.compose(qc_alg, inplace=True)
    cr_m = ClassicalRegister(n, name='measureresult')
    qc.add_register(cr_m)
    qc.measure(range(n), cr_m)
    return transpile(qc, backend, initial_layout=layout, optimization_level=1)


def otem_marginal(data, n_q):
    """Keep only shots where every OT qubit measured 0; return counts of algorithm results."""
    tr = data.testresult.get_bitstrings()
    mr = data.measureresult.get_bitstrings()
    c  = {}
    for t, m in zip(tr, mr):
        if t == '0' * n_q:
            c[m] = c.get(m, 0) + 1
    return c


N_REPS      = 3
all_results = {}

for si, (layout, qc_ot) in enumerate(zip(ALL_LAYOUTS, ot_circuits)):
    print(f"\n=== Layout {si+1}: physical qubits {layout} ===")
    n_q = len(layout)

    ori_qcs = [build_no_otem_circuit(bench[ct], layout, backend)       for ct in ct_list]
    em_qcs  = [build_with_otem_circuit(bench[ct], qc_ot, layout, backend) for ct in ct_list]
    all_qcs = (ori_qcs + em_qcs) * N_REPS

    job = sampler.run(all_qcs, shots=shots)
    print(f'  Layout {si+1} job:', job.job_id())
    res = list(job.result())

    nc  = len(ct_list)
    key = f'l{si+1}'
    all_results[key] = {'ori': {ct: [] for ct in ct_list}, 'em': {ct: [] for ct in ct_list}}
    for rep in range(N_REPS):
        off = rep * 2 * nc
        for ci, ct in enumerate(ct_list):
            o_c = res[off + ci].data.meas.get_counts()
            e_c = otem_marginal(res[off + nc + ci].data, n_q)
            all_results[key]['ori'][ct].append(srate(o_c, n_q))
            all_results[key]['em'][ct].append(srate(e_c, n_q))

    for ct in ct_list:
        o = all_results[key]['ori'][ct]
        e = all_results[key]['em'][ct]
        print(f"  {ct}: ori={np.mean(o):.3f}\u00b1{np.std(o):.3f}  em={np.mean(e):.3f}\u00b1{np.std(e):.3f}")

## 6. Plot — Fig. 20

In [ ]:
layout_keys = list(all_results.keys())
x, w = np.arange(len(ct_list)), 0.18

fig, ax = plt.subplots(figsize=(10, 5))
for si, sk in enumerate(layout_keys):
    for ki, (key, col) in enumerate([('ori', 'tab:red'), ('em', 'tab:green')]):
        means = [np.mean(all_results[sk][key][ct]) for ct in ct_list]
        stds  = [np.std( all_results[sk][key][ct]) for ct in ct_list]
        lbl   = f"{'without' if key == 'ori' else 'with'} OTEM (Layout {si+1})"
        offset = ((2 * si + ki) - (N_LAY - 0.5)) * w
        ax.bar(x + offset, means, w, yerr=[s * 2 for s in stds],
               label=lbl, color=col, alpha=0.9 - 0.4 * si, capsize=4)

ax.set_xticks(x); ax.set_xticklabels(ct_list)
ax.set_ylim(0, 1); ax.set_ylabel('Success Rate')
ax.legend(fontsize=9, ncol=2)
ax.set_title(f'Fig. 20: Five-Qubit Circuits on {backend.name}')
plt.tight_layout()
plt.savefig(f'fig20_5qubit_{backend.name}.pdf', bbox_inches='tight')
plt.show()